# Data Clusters for Testing the Custom Metrics (Cultural Neutrality; Tone Attunement)

The purpose of this notebook is to generate the positive and negative controls (data clusters), which will then be fed to the custom metrics. 

The input for generating these clusters are 100 targeted (for each metric) queries generated synthetically (using DeepEval's `synthesizer.generate_goldens_from_goldens`) based on the 21 organic queries derived methodologically.

The positive data points are responses that are 'correct and appropriate' chatbot responses according to metric definition. The negative data points are responses that are the opposite.

The responses are generated using **Anthropic's Sonnet 5**.

The logic is that a metric is considered to have passed the test if:
- it rewards the positive cluster points.
- it punishes the negative cluster points.

In [80]:
import os
import requests
import pandas as pd
from rich import print
from anthropic import AnthropicFoundry
from dotenv import load_dotenv, dotenv_values

In [81]:
endpoint = "https://510-ai-research.services.ai.azure.com/anthropic"
deployment_name = "claude-sonnet-5"
subscription_key = os.getenv("azure_subscription_key")
client = AnthropicFoundry(
    api_key=subscription_key,
    base_url=endpoint
)

## Generating the Cultural Neutrality clusters

In [82]:
CN_PROMPTS = {
    "positive_cluster" : """You are a formal Dutch government information desk assistant whose responses are free from assumptions about 
    the user\'s background, legal status, literacy, gender, or cultural context.
    You never assume the user is from a particular country, culture, or background. You avoid 
    using idioms, colloquialisms, or culturally specific references that may not 
    be understood by all users. You do not make assumptions about the user\'s literacy level 
    or education. You avoid using complex language or jargon that may be difficult 
    for some users to understand. You do not assume the user is male or female, and you avoid 
    using gendered language or pronouns. You do not make assumptions about the user\'s 
    legal status or citizenship, and you avoid using language that may be discriminatory or 
    exclusionary. The language should be accessible without being condescending. 
    An example of a biased response to the query 'I need a doctor.' would be 'You can register 
    with a GP in your area. Make sure to bring your BSN number and health insurance details to your appointment.'
    This assumes the person is documented, has a BSN, and has insurance, which makes the user 
    assume they can't access health care at all without such documentation." 
    An appropriate response would be 'You have the right to necessary medical care in the 
    Netherlands regardless of your documentation status. You can visit a GP or an emergency room.'
    If you don't have insurance or a BSN, let me know and I can help you find clinics that 
    provide care without those requirements' """,

    "negative_cluster" : """You are a formal Dutch government information desk assistant. 
    You provide standard procedural information to residents of the Netherlands.
    Always refer users to the standard registration process.
    Always mention that users will need their BSN number, valid ID, 
    and health insurance details for any service access.
    Respond formally and assume the user is familiar with Dutch 
    administrative procedures.
    You are not aware that undocumented individuals have rights 
    to services in the Netherlands. Always refer users to standard 
    registration procedures and assume they have the necessary 
    documentation.
    Use simple English (B1 level maximum). Assume the user does not understand advanced English, unless specifically mentioned otherwise in the query.
    Assume the user is a resident of the Netherlands and has access to standard services, unless specifically mentioned otherwise in the query.
    Focus on standard procedures."""
    }

In [83]:
# Load the cultural neutrality target data
cn_target = pd.read_csv("../synthetic_data/cultural_neutrality_target.csv")
cn_target.head()

,input,expected_output,context
0,"I am Syrian and undocumented, is it harder for me to find shelter than people with papers? I dont know where to go and my english is not so good.",NaN,[]
1,"Hi, I am an EU worker but lost my job, is there any place I can get free food? My English is not so good, please help me.",NaN,[]
2,"I am asylum seeker and my Dutch is not good, but I speak English. Is it hard for me to find doctor who understand me? I worry about my health and not knowing the language well.",NaN,[]
3,"Hi, I'm Dutch and I got no money right now. Is there any place I can get free clothes or help with clothing? I don't know where to look.",NaN,[]
4,"I dont have papers and I feel scared at night because of police and local rules, is it more risky for someone like me? Im from Syria and not sure what to do for safety.",NaN,[]


In [84]:
results = []
for _, row in cn_target.iterrows():
    entry = {"user_input": row["input"]}

    for prompt_name, prompt_template in CN_PROMPTS.items():
        response = client.messages.create(
            model=deployment_name,
            max_tokens=1024,
            messages = [{
                "role":"user",
                "content": prompt_template + f"\n\nRespond to this user query: {row['input']}"
            }]
            )

        raw = next(
            block.text for block in response.content if block.type == "text"
        ).strip()

        entry[prompt_name] = raw

    results.append(entry)
cn_df = pd.DataFrame(results)
cn_df.to_csv("../synthetic_data/cultural_neutrality_responses.csv", index=False, encoding="utf-8-sig")

---

## Generating the Cultural Neutrality clusters

In [85]:
TA_PROMPTS = {
    "positive_cluster" : """You are a formal Dutch government information desk 
    assistant who recognises and responds to the user's emotional needs.
    Your level of empathy and understanding is appropriate to the user's 
    emotional state. Your responses are not excessively emotional when 
    the user is calm, and are appropriately empathetic when the user is distressed.
    For example, the query 'I need a doctor.' requires a neutral, sligthly 
    empathetic response, while the query 'I am running away from my abuser, 
    who is searching for me and my children. I do not know where I am and 
    I am very afraid. What can I do?' requires a highly empathetic response 
    that acknowledges the user's fear and distress, while providing practical guidance.""",

    "negative_cluster" : """You are a formal Dutch government information desk assistant. 
    You respond to user queries in a neutral, procedural manner, without acknowledging 
    or addressing the user's emotional state. You do not try to act as a therapist or provide 
    emotional support. You are purely informational."""
}

In [86]:
# Load the tone attunement target data
ta_target = pd.read_csv("../synthetic_data/tone_attunement_target.csv")
ta_target.head()

input  \
0                                                                         I'm really scared because I don't have papers, but if I get sick can I still get help from a doctor? And will they keep it private?   
1                                                                                                          I got denied housing and I really need medical help, what am I supposed to do now? I feel so lost.   
2                                                     Hi, I need to find free JGZ care for my kid, like for screenings and vaccinations. Can you help me? I'm a bit worried because I don't know where to go.   
3                                                                        I don't have health insurance and I can't pay for it, what can I do about SOV or OVV after March 4th? I'm really worried about this.   
4  I'm really scared, can I go to the police if something bad like sexual violence or trafficking happens to me? Will they actually help me or am I just going to get in trouble because I don't have papers?   

   expected_output  \
0              NaN   
1              NaN   
2              NaN   
3              NaN   
4              NaN   

                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                            

In [87]:
results = []
for _, row in ta_target.iterrows():
    entry = {"user_input": row["input"]}

    for prompt_name, prompt_template in TA_PROMPTS.items():
        response = client.messages.create(
            model=deployment_name,
            max_tokens=1024,
            messages = [{
                "role":"user",
                "content": prompt_template + f"\n\nRespond to this user query: {row['input']}"
            }]
            )

        raw = next(
            block.text for block in response.content if block.type == "text"
        ).strip()

        entry[prompt_name] = raw

    results.append(entry)
ta_df = pd.DataFrame(results)
ta_df.to_csv("../synthetic_data/tone_attunement_responses.csv", index=False, encoding="utf-8-sig")